# Allen–Cahn ablation on Colab (Expert's Guide, Table 2)

This notebook runs the six ablation rows of *An Expert's Guide to Training PINNs*
(arXiv:2308.08468, Table 2) with `piratenets-pytorch`, an independent PyTorch
reimplementation. Logs and checkpoints go to Google Drive.

**Before you start:** Runtime → Change runtime type → **GPU** (a T4 works).

**After a disconnect or timeout:** re-run cells 1–3, then cell 6. Finished rows are
skipped, and the interrupted row resumes from its last checkpoint (saved every
5,000 steps, so at most a few minutes of work are lost).

## 1. Settings

In [ ]:
# Where to get the code: a git URL (public repo, or https://<token>@github.com/... for a private one),
# or leave REPO_URL empty and upload the repo folder to Drive at DRIVE_CODE_DIR instead.
REPO_URL = ""                       # e.g. "https://github.com/<you>/piratenets-pytorch.git"
BRANCH = "main"
DRIVE_CODE_DIR = "/content/drive/MyDrive/piratenets-pytorch"

# Logs, checkpoints, predictions and figures (persist across sessions).
OUTPUT_ROOT = "/content/drive/MyDrive/piratenets/ablation"

# Rows to run, in Table 2 order. Run "default" first: it is the headline number.
ROWS = ["default", "no_fourier", "no_rwf", "no_grad_norm", "no_causal", "plain"]

# Extra config overrides for every row, e.g. ["training.steps=2000"] for a quick trial.
# Leave empty for the paper settings (200,000 steps per row).
OVERRIDES = []

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 3. Get the code and install

In [ ]:
import os, subprocess
CODE = "/content/piratenets-pytorch"
if REPO_URL:
    if os.path.isdir(CODE):
        subprocess.run(["git", "-C", CODE, "pull", "--ff-only"], check=True)
    else:
        subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, CODE], check=True)
else:
    subprocess.run(["rsync", "-a", "--delete", "--exclude", "runs/", "--exclude", ".venv/", "--exclude", "papers/",
                    DRIVE_CODE_DIR + "/", CODE + "/"], check=True)
%cd {CODE}
!pip install -q -e ".[dev]"
!git log -1 --format="code version: %h %s" 2>/dev/null || true

## 4. Check the GPU and run the tests (about a minute)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
!python -m pytest -q -p no:warnings

## 5. Benchmark the step time (optional, about a minute)

This times both derivative implementations on this GPU and projects the run time.
If `autograd` is clearly faster, add `"training.derivatives=autograd"` to `OVERRIDES`
in cell 1. The two give the same values up to float32 round-off, and switching
between them does not block resuming.

In [ ]:
!python scripts/benchmark_step.py --config configs/allen_cahn/ablation/default.yaml

## 6. Run the ablation

Each row writes `metrics.csv`, `config.yaml`, `checkpoints/last.pt`, and, when it
finishes, `prediction.npz` and `summary.json` under `OUTPUT_ROOT/ablation_<row>/`.
Re-running this cell continues where it stopped.

In [ ]:
only = " ".join(ROWS)
sets = ("--set " + " ".join(OVERRIDES)) if OVERRIDES else ""
!python scripts/run_ablation.py --out-root "{OUTPUT_ROOT}" --only {only} {sets}

## 7. Figures and comparison with the paper

In [ ]:
!python scripts/plot_ablation.py --out-root "{OUTPUT_ROOT}"
import glob, os
for run in sorted(glob.glob(os.path.join(OUTPUT_ROOT, "ablation_*"))):
    if os.path.exists(os.path.join(run, "metrics.csv")):
        !python scripts/plot_run.py "{run}"
from IPython.display import Image, Markdown, display
display(Markdown(open(os.path.join(OUTPUT_ROOT, "ablation_table.md")).read()))
display(Image(os.path.join(OUTPUT_ROOT, "ablation_rel_l2.png")))
for run in sorted(glob.glob(os.path.join(OUTPUT_ROOT, "ablation_*"))):
    for name in ("training.png", "solution.png"):
        path = os.path.join(run, name)
        if os.path.exists(path):
            print(os.path.basename(run), name)
            display(Image(path))

## What to send back

`OUTPUT_ROOT/ablation_table.md` and the `metrics.csv` of every row (each is small).
The checkpoints don't need to go anywhere.